In [ ]:
# %% [cell 0] environment probe + device selection
# knee submit v1: DICOM decode -> KneeNet 5-fold mean -> submission.csv
import os, sys, time, glob, csv, json
from pathlib import Path

import numpy as np
import torch

INPUT = Path(os.environ.get("KNEE_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("KNEE_WORK", "/kaggle/working"))
WORK.mkdir(parents=True, exist_ok=True)

print("input tree (top 2 levels):")
for p in sorted(INPUT.glob("*")):
    print(" ", p.name, "->", [c.name for c in sorted(p.glob("*"))[:8]])

# GPU roulette guard: a P100 (sm_60) mounts as cuda but this image's torch refuses it.
# A submission must survive that, so probe with a real op and fall back to CPU.
DEV = "cpu"
if torch.cuda.is_available():
    try:
        cap = torch.cuda.get_device_capability(0)
        if cap[0] >= 7:
            (torch.ones(2, device="cuda") * 2).sum().item()
            DEV = "cuda"
        else:
            print(f"GPU {torch.cuda.get_device_name(0)} sm_{cap[0]}{cap[1]} unsupported; using CPU")
    except Exception as e:
        print("CUDA probe failed, using CPU:", e)
torch.set_num_threads(os.cpu_count() or 4)
print("device:", DEV, "| cpus:", os.cpu_count())

In [ ]:
# %% [cell 1] locate inputs
COLS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
        "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
BUCKETS = ["sag_fs", "cor_fs", "axi_fs", "sag_nf", "cor_nf"]
DEPTH, SIZE = 24, 224
# train-set prevalence per column (labels.csv, 4407 studies) — fallback for studies
# with nothing decodable; a dropped row is a failed submission.
PREV = {"ACL": 0.2222, "MCL": 0.1110, "Medial Meniscus": 0.4099, "Lateral Meniscus": 0.2126,
        "Medial OA": 0.2619, "Lateral OA": 0.1984, "PF OA": 0.3640, "Effusion": 0.3961,
        "Synovitis": 0.3664, "Baker's": 0.2020, "Contusion": 0.1807, "Fracture": 0.2072}


def find_one(patterns, what):
    for pat in patterns:
        hits = sorted(glob.glob(str(pat), recursive=True))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(what)


COMP = find_one([INPUT / "*/test_series.csv", INPUT / "*/*/*/test_series.csv",
                 INPUT / "**/test_series.csv"], "competition dir (test_series.csv)").parent
# fold checkpoints live across TWO source kernels (v3a: folds 0-2, v3b: folds 3-4);
# mount layout varies, so glob each fold independently.
FOLD_PTS = {k: find_one([INPUT / f"*/fold{k}.pt", INPUT / f"*/*/*/fold{k}.pt",
                         INPUT / f"**/fold{k}.pt"], f"fold{k}.pt") for k in range(5)}
assert sorted(FOLD_PTS) == [0, 1, 2, 3, 4], FOLD_PTS
print("fold checkpoints:")
for k in range(5):
    print(f"  fold{k}: {FOLD_PTS[k]}")
TRAINPY = find_one([INPUT / "**/train.py"], "train.py")
IMAGES = COMP / "test_series" if (COMP / "test_series").is_dir() else COMP
print("COMP", COMP, "\nTRAINPY", TRAINPY, "\nIMAGES", IMAGES)

test_uids = [r["StudyInstanceUID"] for r in csv.DictReader(open(COMP / "test.csv"))]
series_rows = list(csv.DictReader(open(COMP / "test_series.csv")))
print(len(test_uids), "test studies,", len(series_rows), "series listed")

In [ ]:
# %% [cell 2] choose series per bucket, decode DICOM (the expensive step)
# Decode logic is a verbatim copy of scripts/prep_dicom.py (frozen in the repo),
# written to a module file so ProcessPoolExecutor workers can import it.
WORKER_SRC = r'''
import numpy as np
from pathlib import Path
import cv2
import pydicom


def _slice_order(datasets):
    """Sort by position along the slice normal. InstanceNumber is unreliable in this corpus."""
    try:
        iop = np.asarray(datasets[0].ImageOrientationPatient, dtype=np.float64)
        normal = np.cross(iop[:3], iop[3:])
        keys = [float(np.dot(np.asarray(d.ImagePositionPatient, dtype=np.float64), normal))
                for d in datasets]
    except Exception:
        keys = [float(getattr(d, "InstanceNumber", i)) for i, d in enumerate(datasets)]
    return list(np.argsort(keys))


def _to_hu(ds, arr):
    arr = arr.astype(np.float32)
    slope = float(getattr(ds, "RescaleSlope", 1) or 1)
    inter = float(getattr(ds, "RescaleIntercept", 0) or 0)
    if slope != 1 or inter != 0:
        arr = arr * slope + inter
    if str(getattr(ds, "PhotometricInterpretation", "")) == "MONOCHROME1":
        arr = arr.max() - arr
    return arr


def load_volume(series_dir, size, depth):
    series_dir = Path(series_dir)
    files = sorted(p for p in series_dir.iterdir() if p.is_file() and not p.name.startswith("."))
    if not files:
        raise RuntimeError("empty series dir")

    if len(files) == 1:  # enhanced / multi-frame DICOM: one file, many frames
        ds = pydicom.dcmread(str(files[0]))
        vol = _to_hu(ds, ds.pixel_array)
        if vol.ndim == 2:
            vol = vol[None]
    else:
        dss = [pydicom.dcmread(str(f)) for f in files]
        dss = [d for d in dss if hasattr(d, "pixel_array") or "PixelData" in d]
        order = _slice_order(dss)
        planes = []
        for i in order:
            a = _to_hu(dss[i], dss[i].pixel_array)
            if a.ndim == 3:
                planes.extend(list(a))
            else:
                planes.append(a)
        vol = np.stack(planes)

    n = vol.shape[0]
    idx = np.linspace(0, n - 1, depth).round().astype(int) if n > 1 else np.zeros(depth, int)
    vol = vol[idx]

    out = np.empty((depth, size, size), dtype=np.uint8)
    lo, hi = np.percentile(vol, [1.0, 99.0])
    if hi <= lo:
        hi = lo + 1.0
    for i, plane in enumerate(vol):
        p = np.clip((plane - lo) / (hi - lo), 0, 1)
        out[i] = (cv2.resize(p, (size, size), interpolation=cv2.INTER_AREA) * 255).astype(np.uint8)
    return out


def decode_bucket(job):
    """Try candidate series dirs in order; first successful decode wins."""
    study, bucket, candidates, out_dir, size, depth = job
    errs = []
    for sid, sdir in candidates:
        try:
            vol = load_volume(sdir, size, depth)
            dst = Path(out_dir) / f"{sid}.npy"
            np.save(dst, vol)
            return (study, bucket, sid, 1, "")
        except Exception as e:
            errs.append(f"{sid}: {type(e).__name__}: {e}"[:120])
    return (study, bucket, "", 0, " | ".join(errs)[:300])
'''
(WORK / "dicom_worker.py").write_text(WORKER_SRC)
sys.path.insert(0, str(WORK))
import dicom_worker  # noqa: E402

import multiprocessing as mp  # noqa: E402
from collections import defaultdict  # noqa: E402
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed  # noqa: E402


def bucket_of(plane, fluid):
    p = (plane or "").strip().lower()[:3] or "unk"
    return f"{p}_{'fs' if str(fluid).strip() == '1' else 'nf'}"


t0 = time.time()
# candidates per (study, bucket), ordered by slice-file count desc ("most slices")
cand = defaultdict(list)
missing_dirs = 0
for r in series_rows:
    b = bucket_of(r.get("Anatomical_Plane", ""), r.get("Fluid_Sensitive", ""))
    if b not in BUCKETS:
        continue
    sdir = IMAGES / r["StudyInstanceUID"] / r["SeriesInstanceUID"]
    if not sdir.is_dir():
        hits = glob.glob(str(IMAGES / "**" / r["SeriesInstanceUID"]), recursive=True)
        if not hits:
            missing_dirs += 1
            continue
        sdir = Path(hits[0])
    try:
        nfiles = sum(1 for p in sdir.iterdir() if p.is_file())
    except OSError:
        nfiles = 0
    cand[(r["StudyInstanceUID"], b)].append((nfiles, r["SeriesInstanceUID"], str(sdir)))
if missing_dirs:
    print(f"WARNING: {missing_dirs} listed series have no directory under {IMAGES}")

CACHE = WORK / "cache"
CACHE.mkdir(exist_ok=True)
jobs = []
for (study, b), lst in cand.items():
    lst.sort(key=lambda t: -t[0])
    jobs.append((study, b, [(sid, sdir) for _, sid, sdir in lst], str(CACHE), SIZE, DEPTH))
print(f"{len(jobs)} (study,bucket) decode jobs "
      f"over {sum(len(j[2]) for j in jobs)} candidate series")

chosen = defaultdict(dict)   # study -> bucket -> npy path
fails = []
nworkers = max(1, os.cpu_count() or 2)
try:
    pool = ProcessPoolExecutor(max_workers=nworkers, mp_context=mp.get_context("fork"))
except (ValueError, OSError):
    pool = ThreadPoolExecutor(max_workers=nworkers)
with pool as ex:
    futs = [ex.submit(dicom_worker.decode_bucket, j) for j in jobs]
    for i, f in enumerate(as_completed(futs), 1):
        study, b, sid, ok, err = f.result()
        if ok:
            chosen[study][b] = CACHE / f"{sid}.npy"
        else:
            fails.append((study, b, err))
        if i % 200 == 0:
            print(f"  {i}/{len(jobs)}  elapsed {time.time()-t0:.0f}s", flush=True)

DECODE_SEC = time.time() - t0
print(f"DICOM decode: {DECODE_SEC:.1f}s for {len(jobs)} jobs "
      f"({len(fails)} bucket failures)")
for study, b, err in fails[:10]:
    print("  fail:", study[-12:], b, err[:100])

In [ ]:
# %% [cell 3] model: import KneeNet from the v1 kernel's train.py, 5-fold mean of sigmoids
import importlib.util  # noqa: E402

spec = importlib.util.spec_from_file_location("knee_train", str(TRAINPY))
knee_train = importlib.util.module_from_spec(spec)
spec.loader.exec_module(knee_train)
KneeNet = knee_train.KneeNet
assert knee_train.COLS == COLS

models = []
for k in range(5):
    ck = torch.load(FOLD_PTS[k], map_location="cpu", weights_only=False)
    sd = ck["model"] if isinstance(ck, dict) and "model" in ck else ck
    m = KneeNet("tf_efficientnet_b0", n_buckets=len(BUCKETS), pretrained=False)
    m.load_state_dict(sd)
    m.eval().to(DEV)
    models.append(m)
print(f"loaded {len(models)} folds on {DEV}")


def load_study(uid):
    vols, mask = [], []
    for b in BUCKETS:
        p = chosen.get(uid, {}).get(b)
        v = None
        if p is not None:
            try:
                v = np.load(p)
            except Exception:
                v = None
        if v is None:
            vols.append(np.zeros((DEPTH, SIZE, SIZE), np.uint8))
            mask.append(0.0)
        else:
            if v.shape[0] != DEPTH:
                idx = np.linspace(0, v.shape[0] - 1, DEPTH).round().astype(int)
                v = v[idx]
            vols.append(v)
            mask.append(1.0)
    return np.stack(vols), np.array(mask, np.float32)


t1 = time.time()
BS = 4 if DEV == "cuda" else 2
preds = {}
todo = [u for u in test_uids if chosen.get(u)]
no_pixels = [u for u in test_uids if not chosen.get(u)]
print(f"{len(todo)} studies with pixels, {len(no_pixels)} falling back to prevalence")

with torch.no_grad():
    for s in range(0, len(todo), BS):
        batch = todo[s:s + BS]
        xs, ms = zip(*(load_study(u) for u in batch))
        x = torch.from_numpy(np.stack(xs)).float().div_(255.0).to(DEV)
        m = torch.from_numpy(np.stack(ms)).to(DEV)
        with torch.autocast(DEV, enabled=(DEV == "cuda")):
            p = torch.stack([mod(x, m).float().sigmoid() for mod in models]).mean(0)
        for u, row in zip(batch, p.cpu().numpy()):
            preds[u] = np.clip(row, 0.0, 1.0)
        if (s // BS) % 25 == 0:
            print(f"  {s + len(batch)}/{len(todo)}  elapsed {time.time()-t1:.0f}s", flush=True)

INFER_SEC = time.time() - t1
print(f"inference: {INFER_SEC:.1f}s for {len(todo)} studies on {DEV}")

prev_row = np.array([PREV[c] for c in COLS], np.float32)
for u in no_pixels:
    preds[u] = prev_row.copy()

In [ ]:
# %% [cell 4] write submission.csv + self-checks
sample_path = COMP / "sample_submission.csv"
with open(sample_path) as fh:
    sample_header = fh.readline().rstrip("\n").rstrip("\r")
expected_header = ",".join(["StudyInstanceUID"] + COLS)
assert sample_header == expected_header, f"header drift: {sample_header!r}"

sub_path = WORK / "submission.csv"
with open(sub_path, "w", newline="") as fh:
    w = csv.writer(fh, lineterminator="\n")
    w.writerow(["StudyInstanceUID"] + COLS)
    for u in test_uids:
        row = preds.get(u, prev_row)
        w.writerow([u] + [f"{float(v):.6f}" for v in row])

# validate
import pandas as pd  # noqa: E402
sub = pd.read_csv(sub_path)
assert list(sub.columns) == ["StudyInstanceUID"] + COLS
assert len(sub) == len(test_uids), (len(sub), len(test_uids))
V = sub[COLS].to_numpy()
assert np.isfinite(V).all(), "NaN/inf in submission"
assert (V >= 0).all() and (V <= 1).all(), "values outside [0,1]"
with open(sub_path) as fh:
    assert fh.readline().rstrip("\n") == sample_header, "header mismatch vs sample"
print(f"submission.csv OK: {len(sub)} rows")
print("per-column mean:", dict(zip(COLS, V.mean(0).round(4))))
print("per-column std :", dict(zip(COLS, V.std(0).round(4))))
if len(sub) > 1 and V.std() < 1e-6:
    print("WARNING: predictions are constant — model likely never ran")
print(f"timings: decode={DECODE_SEC:.0f}s infer={INFER_SEC:.0f}s device={DEV}")
print(sub.head(3).to_string(index=False))